## Task 6: Hit and Run Accident Investigation

On 11.08.2010, there was a hit-and-run accident. The police need to find out where the vehicle with body part number `K5-112-1122-79` was registered.

**Strategy:**
1. Identify which column contains body part IDs by checking a vehicle parts file
2. Search all `Bestandteile_Fahrzeuge` files for the body part number to find the vehicle ID
3. Cross-check the vehicle in the corresponding `Fahrzeuge` file to verify plausibility
4. Look up the vehicle ID in the registration table to find the municipality
5. Retrieve geographic details from the geodata table

Checking the structure of a vehicle parts file to identify the relevant column for body parts.

In [33]:
import pandas as pd

df = pd.read_csv("data/Fahrzeug/Bestandteile_Fahrzeuge_OEM1_Typ11.csv", sep = ";").drop(columns=["Unnamed: 0"])

df.head()

,ID_Karosserie,ID_Schaltung,ID_Sitze,ID_Motor,ID_Fahrzeug
0,K4-112-1121-3,K3SG1-105-1051-32,K2LE1-109-1091-2,K1BE1-101-1011-7,11-1-11-1
1,K4-112-1121-4,K3SG1-105-1051-141,K2ST1-109-1092-5,K1BE1-101-1011-12,11-1-11-2
2,K4-112-1121-7,K3SG1-105-1051-106,K2ST1-109-1092-57,K1BE1-101-1011-38,11-1-11-3
3,K4-112-1121-9,K3SG1-105-1051-21,K2ST1-109-1092-91,K1BE1-101-1011-97,11-1-11-4
4,K4-112-1121-11,K3SG1-105-1051-59,K2ST1-109-1092-4,K1BE1-101-1011-65,11-1-11-5


The body part IDs are stored in the `ID_Karosserie` column. All four vehicle parts files are searched for the given body part number.

In [34]:
body_part_number = "K5-112-1122-79"

bestandteile_files = ["OEM1_Typ11.csv", "OEM1_Typ12.csv", "OEM2_Typ21.csv", "OEM2_Typ22.csv"]

# Check the already loaded file first
check = df[df["ID_Karosserie"] == body_part_number]

if len(check) == 0:
    for i in bestandteile_files[1:]:
        body_parts = pd.read_csv("data/Fahrzeug/Bestandteile_Fahrzeuge_" + i, sep=";").drop(columns=["Unnamed: 0"])
        check = body_parts[body_parts["ID_Karosserie"] == body_part_number]
        if len(check) > 0:
            break

display(check)

,ID_Karosserie,ID_Schaltung,ID_Sitze,ID_Motor,ID_Fahrzeug
81,K5-112-1122-79,K3SG1-105-1051-129,K2ST1-109-1092-519,K1BE1-104-1041-409,12-1-12-82


The body part was found in `OEM1_Typ12`. The vehicle ID is `12-1-12-82`. To verify, the vehicle is checked in the corresponding `Fahrzeuge_OEM1_Typ12.csv` file.

In [35]:
id_vehicle = check["ID_Fahrzeug"].values[0]

# Cross-check in vehicle file
vehicles = pd.read_csv("data/Fahrzeug/Fahrzeuge_OEM1_Typ12.csv", sep=";").drop(columns=["Unnamed: 0", "X1"])

check_vehicle = vehicles[vehicles["ID_Fahrzeug"] == id_vehicle]
display(check_vehicle)

,ID_Fahrzeug,Produktionsdatum,Herstellernummer,Werksnummer,Fehlerhaft,Fehlerhaft_Datum,Fehlerhaft_Fahrleistung
81,12-1-12-82,2008-11-21,1,12,0,NaN,0


The vehicle was produced on 2008-11-21, before the accident date (2010-08-11). The vehicle ID is looked up in the registration table.

In [36]:
registration = pd.read_csv("data/Zulassungen/Zulassungen_alle_Fahrzeuge.csv", sep=";").drop(columns=["Unnamed: 0"])

find_registration = registration[registration["IDNummer"] == id_vehicle]
display(find_registration)

,IDNummer,Gemeinden,Zulassung
223,12-1-12-82,ASCHERSLEBEN,2009-01-02


The vehicle is registered in ASCHERSLEBEN. The geodata table provides additional location details.

In [37]:
vehicle_registered_in = find_registration["Gemeinden"].values[0]

geodata = pd.read_csv("data/Geodaten/Geodaten_Gemeinden_v1.2_2017-08-22_TrR.csv", sep=";").drop(columns=["Unnamed: 0", "X"])
aschersleben_details = geodata[geodata["Gemeinde"] == vehicle_registered_in]
display(aschersleben_details)

print(f"\nThe vehicle {id_vehicle} with body part {body_part_number} "
      f"is registered in {vehicle_registered_in}")

,Postleitzahl,Gemeinde,Laengengrad,Breitengrad
393,6449,ASCHERSLEBEN,"11,455041","51,756034"



The vehicle 12-1-12-82 with body part K5-112-1122-79 is registered in ASCHERSLEBEN


**Result:** The vehicle with body part number `K5-112-1122-79` is registered in `ASCHERSLEBEN` (postal code 6449).